## Loop

`Conditional_Agent.ipynb` branched, but every path still moved *forward* — no node ever ran twice. This notebook introduces a real **cycle**: the `random` node's conditional edge can route back to `random` itself, so the same node re-executes until a stop condition is met. This is the pattern behind `LoopAgent`-style "repeat until done" workflows — and also the pattern that, if the stop condition is wrong, produces exactly the kind of "runs forever" behavior we debugged in `Conditional_Agent.ipynb`. Here the stop condition is correct, so it terminates cleanly.

In [ ]:
from langgraph.graph import StateGraph, END
import random
from typing import Dict, List, TypedDict

Only `END` is imported this time (no `START`) — this notebook uses `set_entry_point("greeting")` instead of `add_edge(START, "greeting")`; both are equivalent ways to mark the entry node, just different API styles. `random` is plain Python `random`, used to generate demo data each loop iteration.

In [ ]:
class AgentState(TypedDict):
    name: str
    number: List[int]
    counter: int

def greeting_node(state: AgentState) -> AgentState:
    """Greeting Node which says hi yo yhe person"""
    state["name"] = f"Hi there, {state["name"]}"
    state["counter"] = 0
    return state

def random_node(state:AgentState) -> AgentState:
    """Generate a random number from 0 to 10"""
    state["number"].append(random.randint(0,10))
    state["counter"]+= 1

    return state


def should_continue(state: AgentState) -> AgentState:
    """Function to decide what to do next"""
    if state["counter"] < 5:
        print("Entering loop", state["counter"])
        return "loop"
    else:
        return "exit"


**Where the loop logic actually lives:**
- `greeting_node` runs once, at the start, and resets `counter` to `0` — so whatever `counter` value you pass into `invoke` gets immediately overwritten here.
- `random_node` does the repeated work (appends a random number, increments `counter`) but has no stop condition of its own — it always just does its thing and returns.
- `should_continue` is the router function (same role as `decide_next_node` in `Conditional_Agent.ipynb`): it inspects `counter` and returns `"loop"` or `"exit"`. **This is the only place the loop can stop** — if this logic were wrong (e.g. always returning `"loop"`), `random_node` would run forever.

The `print("Entering loop", ...)` is just a debug trace so you can watch iterations happen when you run the invoke cell below.

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("greeting", greeting_node)
graph.add_node("random", random_node)
graph.add_edge("greeting", "random")

graph.add_conditional_edges(
    "random",
    should_continue,
    {
        "loop": "random",
        "exit": END
    }

)

graph.set_entry_point("greeting")

app = graph.compile()


**The self-loop:** `add_conditional_edges("random", should_continue, {"loop": "random", "exit": END})` attaches the conditional edges directly to `random` itself — when `should_continue` returns `"loop"`, the path map sends execution right back into `random`. That's the whole cycle: no separate passthrough "router" node is needed here (unlike `Conditional_Agent.ipynb`'s `router`/`router2`), because `random` is both the worker node *and* the thing whose output decides whether to repeat.

In [ ]:
from IPython.display import Image,display
display(Image(app.get_graph().draw_mermaid_png()))

The diagram should show an arrow looping from `random` back to itself — visually distinct from every earlier notebook's diagrams, where arrows only ever pointed forward.

In [ ]:
app.invoke({"name": "Sam", "number": [], "counter": -1})

Trace: `counter=-1` in the input is discarded immediately — `greeting_node` resets it to `0`. Then `random → should_continue` fires repeatedly: `counter` goes 1, 2, 3, 4 ("Entering loop" prints for each, since `< 5`), then on the 5th pass `counter == 5` so `should_continue` returns `"exit"`. That's **5** total runs of `random_node` (1 unconditional + 4 looped), matching the 5-element `number` list in the result.

This is also a live example of the recursion-limit safety net mentioned above: LangGraph caps graphs at 25 steps by default (`GraphRecursionError` if exceeded), so even a genuinely broken `should_continue` wouldn't hang your kernel forever — it would error out after 25 steps instead. That default is a debugging aid, not a substitute for a correct stop condition.